In [43]:
# --------------------- Import Libraries ---------------------
import os
import sys
import yaml
from pathlib import Path

import scanpy as sc
import anndata as ad
import squidpy as sq
import seaborn as sns
import pandas as pd
import numpy as np
import networkx as nx
import scipy.sparse as sp
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj

from vqniche.utils.parse_test_configs import *
from vqniche.initializers.initialize import *
from vqniche import metrics
from vqniche.utils.type_conversions import *

## Functions

### Initialize

In [ ]:
def build_inference_adata(
        inference_data: Dict,
        dataset_blob: InMemoryDataset,
    ) -> ad.AnnData:
    """
    Build an AnnData object from inference data dictionary and dataset blob.
    
    Parameters
    ----------
    inference_data : Dict
        Dictionary containing inference results with keys:
        - 'X': Input features (torch.Tensor)
        - 'Y_cell_type': Cell type labels as one-hot vectors (torch.Tensor)
        - 'Y_niche_type': Niche type labels as one-hot vectors (torch.Tensor)
        - 'edge_index': Edge indices (torch.Tensor)
        - 'H_latent': Latent representations (torch.Tensor)
        - 'X_hat': Reconstructed features (torch.Tensor)
        - 'H_adj': Adjacency representations (torch.Tensor)
        - 'H_quantized': Quantized representations (torch.Tensor, optional)
        - 'Indices': Quantization indices (torch.Tensor, optional)
    dataset_blob
        Dataset blob object containing label categories and gene panel information.
    
    Returns
    -------
    ad.AnnData
        AnnData object with inference results stored in appropriate slots.
    """
    # Create AnnData object with input features
    adata = ad.AnnData(X=inference_data['X'].cpu().numpy())
    
    # Set gene panel from dataset blob
    adata.var = dataset_blob.gene_panel
    
    # Convert one-hot labels to string labels
    adata.obs['cell_type'] = torch_one_hot_to_label_name(
        inference_data['Y_cell_type'].cpu(),
        dataset_blob.label_categories['cell_type']
    )
    
    adata.obs['niche_type'] = torch_one_hot_to_label_name(
        inference_data['Y_niche_type'].cpu(),
        dataset_blob.label_categories['niche_type']
    )
    
    # Store embeddings in obsm
    adata.obsm['H_latent'] = inference_data['H_latent'].cpu().numpy()
    adata.obsm['X_hat'] = inference_data['X_hat'].cpu().numpy()
    adata.obsm['H_adj'] = inference_data['H_adj'].cpu().numpy()
    
    # Convert edge index to adjacency matrix and store in obsp
    edge_index = inference_data['edge_index']
    adj_matrix = to_dense_adj(edge_index)[0]
    sparse_adj = sp.csr_matrix(adj_matrix.cpu().numpy())
    adata.obsp['spatial_connectivities'] = sparse_adj
    
    # Add H_quantized and Indices only if they exist
    if 'H_quantized' in inference_data:
        adata.obsm['H_quantized'] = inference_data['H_quantized'].cpu().numpy()
    
    if 'Indices' in inference_data:
        adata.obs['Indices'] = inference_data['Indices'].cpu().numpy()
    
    return adata

In [32]:
def load_everything(
        config: Dict,
    ):
    """
    Load all the necessary components for a model run.
    
    Parameters
    ----------
    config: Dict
        A dictionary containing the configuration parameters for the model run.
    
    Returns
    -------
    - None
    """
    # --------------------- Determinism Settings ---------------------
    pl.seed_everything(config['experiment']['seed'])

    # --------------------- Dataset ---------------------
    dataset_blob = initialize_dataset_blob(config)

    # --------------------- Databatch ---------------------
    data_batch = initialize_databatch(
                    config=config,
                    dataset_blob=dataset_blob,
                )
    
    # --------------------- Dataloader ---------------------
    datamodule_batch = initialize_datamodule(
                            config=config,
                            data=data_batch,
                        )

    # --------------------- Model ---------------------
    Model = set_model_class(config['model']['model_name'])
    if model_ckpt_fname is None:
        model_ckpt_fname = find_best_checkpoint(config['experiment']['wandb_run_dir'])
    model = Model.load_from_checkpoint(model_ckpt_fname)
    
    # --------------------- Trainer ---------------------
    strategy = "ddp_notebook"
    # strategy = "ddp"
    
    trainer = pl.Trainer(
                    accelerator="auto",
                    devices="auto",
                    deterministic=True,
                    logger=False,
                    callbacks=False,
                    strategy=strategy,
                    max_epochs=config['trainer']['max_epochs'],
                    enable_checkpointing=False,
                    num_sanity_val_steps=0,
                    enable_progress_bar=False,
                    enable_model_summary=True,
                )
    
    inference_data = model.collect_inference_data(
                    datamodule_batch.infer_dataloader()
                )
    
    adata = build_inference_adata(
            inference_data=inference_data,
            dataset_blob=dataset_blob,
        )
    
    
    return dataset_blob, \
        data_batch, \
        datamodule_batch, \
        model, \
        trainer, \
        inference_data, \
        adata

In [1]:
def compute_umap(
        adata: ad.AnnData,
        embedding_key: str = 'X',
    ) -> ad.AnnData:
    """
    This function computes the nearest neighbor distance matrix and neighborhood graph,
    and embeds the neighborhood into 2D using UMAP.
    
    Parameters
    ----------
    adata : ad.AnnData
        AnnData object with the data to be processed.
    embedding_key : str
        Key in adata.obsm to use for the embedding.
    
    Returns
    -------
    adata : ad.AnnData
        AnnData object with the UMAP embedding added to adata.obsm.
    """
    # compute nearest neighbor distance matrix and neighborhood graph
    sc.pp.neighbors(
        adata=adata,
        n_neighbors=15, # default is 15
        knn=True, # default is True
        random_state=42,
        n_jobs=1,
        use_rep=embedding_key,
        key_added=f'{embedding_key}_neighbors',
    )
    #  The neighbors data is added to .uns[key_added], distances are stored in .obsp[key_added+'_distances'] and connectivities in .obsp[key_added+'_connectivities'].
    
    # embed the neighborhood into 2D using UMAP
    sc.tl.umap(
        adata=adata,
        min_dist=0.5, # default is 0.5
        spread=1.0, # default is 1.0
        alpha=1.0, # default is 1.0
        gamma=1.0, # default is 1.0
        negative_sample_rate=5, # default is 5
        random_state=42,
        neighbors_key=f'{embedding_key}_neighbors',
        key_added=f'{embedding_key}_umap',
    )
    # The embedding is stored as obsm[key_added] and the the parameters in uns[key_added].

    return adata

SyntaxError: incomplete input (2448758607.py, line 6)

### Model Training

In [ ]:
def read_train_val_metrics(
        wandb_run_dir: Path = None,
    ) -> pd.DataFrame:
    """
    This function reads all metrics logged to output log file during each training epoch and returns them as a Pandas DataFrame.
    
    Parameters
    ----------
    wandb_run_dir : str | Path
        Path to the wandb run directory containing the output log file.
        
    Returns
    -------
    df: pd.DataFrame
        DataFrame containing all metrics logged during each training epoch.
        
    Notes:
    ------
    - This expects that the log file is named 'train_val_epoch_metrics.csv' and is located in the 'files' directory of the wandb run directory.
    - The log file is expected to have loss terms and accuracies as headings and their corresponding values as rows.
    - All other lines in the log file are ignored.
    - The metrics are various losses and accuracies logged during training and validation epochs.
    """
    logfile = wandb_run_dir / 'files' / 'train_val_epoch_metrics.csv'
    raw_df = pd.read_csv(logfile)

    # replace 'nan' strings with actual NaN values
    raw_df.replace('nan', np.nan, inplace=True)

    return raw_df

### Model Testing

### Plotting

In [ ]:
def plot_train_val_metrics(
        loss_df: pd.DataFrame,
        test_acc: float = None,
    ):
    """
    This function plots all metrics logged during training and validation epochs.
    
    Parameters
    ----------
    loss_df : pd.DataFrame
        DataFrame containing all metrics logged during each training epoch.
    test_acc : float
        Test accuracy of the model.
    
    Returns
    -------
    None
    """
    # Melt the dataframe to have a 'Mode' column for 'Train' and 'Val'
    df = loss_df.melt(id_vars=['epoch'], 
                        value_vars=['train_cross_entropy', 'val_cross_entropy', 
                                    'train_mse_attribute_reconstruction', 'val_mse_attribute_reconstruction', 
                                    'train_mse_adjacency_reconstruction', 'val_mse_adjacency_reconstruction', 
                                    'train_mse_commitment_loss', 'val_mse_commitment_loss', 
                                    'train_l2_codebook_loss', 'val_l2_codebook_loss', 
                                    'train_loss', 'val_loss'],
                        var_name='Loss Term', value_name='Value')

    # Create 'Mode' column
    df['Mode'] = df['Loss Term'].apply(lambda x: 'Train' if 'train' in x else 'Val')

    # Simplify 'Loss Term' column
    df['Loss Term'] = df['Loss Term'].apply(lambda x: x.replace('train_', '').replace('val_', ''))

    # Replace metric values with proper names
    metric_names = {
        'cross_entropy': 'Cross Entropy Loss',
        'mse_attribute_reconstruction': 'MSE Attr. Reconstr.',
        'mse_adjacency_reconstruction': 'MSE Adj. Reconstr.',
        'mse_commitment_loss': 'MSE Commit Loss',
        'l2_codebook_loss': 'L2 Codebook Loss',
        'loss': 'Total Loss'
    }

    df['Loss Term'] = df['Loss Term'].map(metric_names)

    # Create a column to indicate if the metric value is NaN
    df['isNaN'] = df['Value'].isna()

    # Group by 'Loss Term' and replace NaN with 1.1 * max value for each group
    df['Value'] = df.groupby('Loss Term')['Value'].transform(lambda x: x.fillna(1.1 * x.max()))
    display(df)
        
    loss_terms = df['Loss Term'].unique()

    fig, axes = plt.subplots(2, 3, figsize=(12,8))

    for ax, loss_term in zip(axes.flatten(), loss_terms):
        sns.lineplot(data=df[df['Loss Term'] == loss_term], x='epoch', y='Value', hue='Mode', style='isNaN', markers=True, dashes=False, ax=ax)

        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove()

        ax.set_xlabel('Epoch')
        ax.set_ylabel(loss_term)
        ax.set_title(loss_term)
    
    fig.legend(
        handles,
        labels,
        bbox_to_anchor=(0.5,-0.08),
        loc='lower center',
        ncol=2,
    )

    fig.suptitle(f"Test Accuracy: {test_acc:.2f}", fontsize=16)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_umap(  
        adata: ad.AnnData,
        embedding_keys: list[str],
        label_key: str = 'cell_type',
    ):
    """
    This function plots the UMAP embedding of the data.
    
    Parameters
    ----------
    adata : ad.AnnData
        AnnData object with the data to be processed.
    embedding_keys : list[str]
        Keys in adata.obsm to use for the embedding.
    label_key : str
        Key in adata.obs to use for the label.
    
    Returns
    -------
    None
    """
    fig, axes = plt.subplots(1, len(embedding_keys), figsize=(12,4))
    for ax, embedding_key in zip(axes, embedding_keys):
        sc.pl.umap(
            adata=adata,
            color=label_key,
            layer=f'{embedding_key}_umap',
            neighbors_key=f'{embedding_key}_neighbors',
            show=False,
            ax=ax,
        )
        ax.set_title(embedding_key)

    plt.tight_layout()
    plt.show()

## Analysis

### xhs1000-39b_1p (batch 11)

In [32]:
# set dataset name and batch id
dataset_name = 'xhs1000-39b_1p'
batch_id = 11

# set wandb run id
wandb_run_id = 'run-20250629_215226-2vypyus4'
config = collect_test_configs(
    wandb_run_dir=wandb_run_id,
)

# load everything
dataset_blob, \
data_batch, \
datamodule_batch, \
model, \
trainer, \
inference_data = load_everything(config)

In [ ]:
adata = build_inference_adata(
    inference_data=inference_data,
    dataset_blob=dataset_blob,
)

In [ ]:
compute_umap(
    adata=adata,
    embedding_keys=['X'],
    label_key='cell_type',
)

In [ ]:
plot_umap(
    adata=adata,
    embedding_keys=['X'],
    label_key='cell_type',
)